In [1]:
here::i_am("rna/regression/regress_variables.R")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(Seurat))

# Multicore
BPPARAM <- BiocParallel::bpparam()
BPPARAM$workers = 24

# Multi core using future - built in to seurat
plan("multicore", workers = 24)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

# Set args
args = list()
args$rna_metadata = file.path(io$basedir, 'results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz')
args$rna_sce = file.path(io$basedir, 'processed/rna/SingleCellExperiment.rds')
args$rna_nfeatures = 4000
args$vars_to_regress = c("nFeature_RNA", "nCount_RNA", "mitochondrial_percent_RNA", "ribosomal_percent_RNA")
args$regression_out = file.path(io$basedir, 'results/rna/regression/')
dir.create(args$regression_out, recursive=TRUE, showWarnings =FALSE)

# Load metadata
metadata_rna <- fread(args$rna_metadata) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE] %>% 
  .[,exp := str_replace_all(sample, opts$sample2exp)]

# Load sce
rna.sce <- load_SingleCellExperiment(args$rna_sce, normalise = TRUE, cells = metadata_rna$cell)

# Add sample metadata to the colData of the SingleCellExperiment
colData(rna.sce) <- metadata_rna %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(rna.sce),] %>% DataFrame()

# Filter features manually
rna.sce <- rna.sce[grep("*Rik|^Gm|^Mt-|^Rps|^Rpl|^Olfr",rownames(rna.sce), invert=T),]

# Remove genes with very low variance
gene_vars = rowVars(logcounts(rna.sce))
names(gene_vars) = rownames(rna.sce)
keep_genes = names(gene_vars[gene_vars>0.1])
rna.sce = rna.sce[keep_genes, ]

# Regress out variables 
logcounts_regressed.mtx <- RegressOutMatrix(
    mtx = logcounts(rna.sce),
    covariates = metadata_rna[,args$vars_to_regress,with=F]
  )


fwrite(as.data.table(logcounts_regressed.mtx, keep.rownames=T), file.path(args$regression_out,"logcounts_regressed_mtx_v2.txt.gz"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code



In [7]:
args$regression_out

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/rna/regression/"

In [ ]:
logcounts_regressed = fread("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/rna/regression/logcounts_regressed_mtx_v2.txt.gz")


In [2]:
logcounts_regressed = fread(file.path(args$regression_out,"logcounts_regressed_mtx_v2.txt.gz"))

In [4]:
logcounts_regressed = test

In [5]:
logcounts_regressed.mtx = logcounts_regressed %>% as.data.table() %>% tibble::column_to_rownames('rn') %>% as.matrix()

In [6]:
test2[1:5, 1:5]

,1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1,1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1,1A_Eo_DEG_G9_day3#AAACATGCAAGTAAGC-1,1A_Eo_DEG_G9_day3#AAACATGCAATATACC-1,1A_Eo_DEG_G9_day3#AAACATGCAATCCTGA-1
Xkr4,0.77238353,1.4823085,2.58460972,2.1789049,0.71431931
Mrpl15,0.37392930,0.9050925,-1.03022064,1.1834188,0.46455369
Lypla1,0.06255539,-0.5070801,0.85362192,-0.4607898,-0.46395751
Tcea1,-0.36605295,-0.1437590,0.03645313,1.2001068,0.06852529
Rgs20,0.29379828,-0.2868279,-0.24199438,-0.2808857,-0.27010769


In [3]:
head(test)

rn,1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1,1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1,1A_Eo_DEG_G9_day3#AAACATGCAAGTAAGC-1,1A_Eo_DEG_G9_day3#AAACATGCAATATACC-1,1A_Eo_DEG_G9_day3#AAACATGCAATCCTGA-1,1A_Eo_DEG_G9_day3#AAACATGCACTTAGGC-1,1A_Eo_DEG_G9_day3#AAACATGCAGCCAGAA-1,1A_Eo_DEG_G9_day3#AAACATGCATCACTTC-1,1A_Eo_DEG_G9_day3#AAACCAACAGAGAGCC-1,⋯,rv_eo_deg_day4_dtag#TTTGCGGAGCTCGCTT-1,rv_eo_deg_day4_dtag#TTTGCGGAGTTGTCTT-1,rv_eo_deg_day4_dtag#TTTGGTAAGAACCTGT-1,rv_eo_deg_day4_dtag#TTTGGTAAGCTCCTAC-1,rv_eo_deg_day4_dtag#TTTGGTAAGTTTGAGC-1,rv_eo_deg_day4_dtag#TTTGTCTAGTTAGTGC-1,rv_eo_deg_day4_dtag#TTTGTGAAGCACTAAC-1,rv_eo_deg_day4_dtag#TTTGTGGCACAAAGAC-1,rv_eo_deg_day4_dtag#TTTGTGTTCATTTGTC-1,rv_eo_deg_day4_dtag#TTTGTTGGTACTTAGG-1
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Xkr4,0.77238353,1.4823085,2.58460972,2.1789049,0.71431931,1.7677688,1.7079066,-0.7626992,-0.4946380,⋯,-0.2569561,-0.5169786,0.03904818,-0.6024997,-0.5893270,-0.5495662,0.16125701,-0.5711478,-0.4924393,0.06370763
Mrpl15,0.37392930,0.9050925,-1.03022064,1.1834188,0.46455369,1.2191161,0.8354437,0.8984897,1.2383216,⋯,1.7874184,0.2846023,-1.32362585,0.3614363,0.4290162,-1.0508667,-0.46859026,0.3810330,-0.3210061,-0.84107948
Lypla1,0.06255539,-0.5070801,0.85362192,-0.4607898,-0.46395751,0.3758642,-0.3620927,0.8406584,1.7161833,⋯,-0.3911510,0.2860222,-0.32010032,0.3158374,0.3821350,0.8648815,-0.50995998,0.7945376,0.4861362,-0.51955326
Tcea1,-0.36605295,-0.1437590,0.03645313,1.2001068,0.06852529,0.1222768,-1.1494347,0.4242810,0.2211486,⋯,-1.1487955,0.6323800,2.27396401,-1.3773699,0.4848500,-0.5253118,0.24288977,0.6866709,0.2057996,-0.71839009
Rgs20,0.29379828,-0.2868279,-0.24199438,-0.2808857,-0.27010769,-0.2980903,-0.1415710,-0.3312593,-0.2384216,⋯,-0.1180276,-0.2880416,-0.11114615,-0.3116707,-0.2992365,1.0841246,-0.33099954,1.6647748,0.2456364,0.71156681
Atp6v1h,-0.72216203,-1.2334483,0.29836107,-1.1839937,-1.14700459,-0.3512598,1.1984358,-1.3395874,0.4109791,⋯,-0.7157266,0.4790894,-0.52080002,0.5248627,0.9659595,1.8497777,0.05403841,0.1346740,0.6148346,0.21989681
